In [0]:
client = mlflow.tracking.MlflowClient()
metrics = client.get_metric_history(RUN_ID, "val_loss")
for m in metrics:
    print(f"epoch {m.step}: {m.value:.4f}")

In [0]:
dbutils.library.restartPython()

In [0]:
%pip install tensorflow

In [0]:
import json, re
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from pyspark.sql import functions as F
import mlflow
import mlflow.tensorflow

RUN_ID = "cf085433cd7f4585bed5366959298606"

# ── Load model ────────────────────────────────────────────────────────────────
lm_model = mlflow.tensorflow.load_model(f"runs:/{RUN_ID}/model")

client  = mlflow.tracking.MlflowClient()
params  = client.get_run(RUN_ID).data.params
SEQ_LEN = int(params["SEQ_LEN"])
print("SEQ_LEN:", SEQ_LEN)

# ── Define standardize first — needed before restoring vectorizer ─────────────
vocab = json.loads(mlflow.artifacts.load_text(f"runs:/{RUN_ID}/vocab.json"))
id_to_token = np.array(vocab)

def custom_standardize(text):
    text = tf.strings.lower(text)
    text = tf.strings.regex_replace(text, r"(<[^>]+>)", r" \1 ")
    text = tf.strings.regex_replace(text, r"[^\w\s<>]", " ")
    text = tf.strings.regex_replace(text, r"\s+", " ")
    return tf.strings.strip(text)

vectorizer = layers.TextVectorization(
    max_tokens=len(vocab),
    standardize=custom_standardize,
    split="whitespace",
    output_mode="int",
)

# vocab[0] = '' (padding), vocab[1] = '[UNK]'
# set_vocabulary expects only the real tokens, no reserved slots
vectorizer.set_vocabulary(vocab[2:])

VOCAB_SIZE = len(vocab)
print("Vocab size:", VOCAB_SIZE)

# sanity check
test_ids    = vectorizer(tf.constant(["<TITLE> batman <PLOT> fights the joker"]))[0].numpy()
test_tokens = [vocab[i] for i in test_ids if i > 0]
print("Sanity check:", test_tokens)
# Expected: ['<title>', 'batman', '<plot>', 'fights', 'the', 'joker']

In [0]:
@tf.function(input_signature=[tf.TensorSpec(shape=(1, None), dtype=tf.int32)])
def fast_predict(x):
    return lm_model(x, training=False)

def sample_from_logits(logits, temperature=0.8, top_k=50):
    logits = tf.cast(logits, tf.float32)
    logits = logits / max(float(temperature), 1e-6)

    if top_k is not None and top_k > 0:
        # FIX: clamp k to actual vocab size to avoid shape errors
        k_actual = min(int(top_k), tf.shape(logits)[0].numpy())
        values, _ = tf.math.top_k(logits, k=k_actual)
        cutoff = values[-1]
        logits = tf.where(logits < cutoff, tf.constant(-1e10, dtype=logits.dtype), logits)

    probs   = tf.nn.softmax(logits)
    next_id = int(tf.random.categorical(tf.math.log([probs]), 1)[0, 0])
    return next_id

def detokenize(ids):
    toks = id_to_token[ids]
    toks = [t for t in toks if t not in ("", "[UNK]")]
    return " ".join(toks)

def generate(prompt: str, max_new_tokens: int = 120,
             temperature: float = 0.7, top_k: int = 50) -> str:
    ids = vectorizer(tf.constant([prompt]))[0]
    ids = tf.boolean_mask(ids, ids > 0).numpy().tolist()

    prompt_len = len(ids)

    for _ in range(max_new_tokens):
        window = ids[-SEQ_LEN:]
        if len(window) < SEQ_LEN:
            window = [0] * (SEQ_LEN - len(window)) + window

        x      = tf.constant([window], dtype=tf.int32)
        logits = fast_predict(x)           # (1, SEQ_LEN, vocab_size)
        next_id = sample_from_logits(logits[0, -1], temperature=temperature, top_k=top_k)
        ids.append(next_id)

    return detokenize(ids[prompt_len:])

In [0]:
MOVIE_TABLE = "default.wiki_movie_plots_deduped"
movies_df = spark.table(MOVIE_TABLE)

STOPWORDS = {
    "the", "and", "for", "with", "was", "are", "this", "that",
    "have", "from", "not", "who", "what", "when", "where", "about",
    "movie", "film", "show", "did", "does", "can", "tell", "give",
}


def retrieve_movies(question: str, k: int = 5, df=movies_df):
    q      = (question or "").lower().strip()
    # FIX: filter stopwords so they don't pollute scores
    tokens = [
        t for t in re.findall(r"[a-z0-9]+", q)
        if len(t) >= 3 and t not in STOPWORDS
    ]
    tokens = list(dict.fromkeys(tokens))  # unique, preserve order

    hay = F.lower(F.concat_ws(" ",
        F.coalesce(F.col("Title"),    F.lit("")),
        F.coalesce(F.col("Plot"),     F.lit("")),
        F.coalesce(F.col("Genre"),    F.lit("")),
        F.coalesce(F.col("Director"), F.lit("")),
        F.coalesce(F.col("Cast"),     F.lit("")),
    ))

    if not tokens:
        return df.limit(k)

    score = None
    for t in tokens:
        hit   = F.when(hay.contains(t), F.lit(1)).otherwise(F.lit(0))
        score = hit if score is None else (score + hit)

    return (
        df.withColumn("_score", score)
        .where(F.col("_score") > 0)
        .orderBy(F.col("_score").desc())
        .limit(k)
    )


In [0]:
def row_to_prompt(r) -> str:
    def ct(v):
        if v is None: return ""
        return re.sub(r"\s+", " ", str(v).replace("\u00a0", " ")).strip()

    return (
        f"<TITLE> {ct(r['Title'])} "
        f"<YEAR> {ct(r['Release Year'])} "
        f"<DIRECTOR> {ct(r['Director'])} "
        f"<GENRE> {ct(r['Genre'])} "
        f"<PLOT> {ct(r['Plot'])[:300]} "   # truncate so prompt fits in SEQ_LEN
    ).strip()

In [0]:
def chat(question: str, max_new_tokens: int = 120,
         temperature: float = 0.7, top_k: int = 10) -> str:
    hits = retrieve_movies(question, k=3).toPandas()

    if hits.empty:
        return "No matching movies found."

    top    = hits.iloc[0]
    prompt = row_to_prompt(top)

    print(f"[Retrieved: {top['Title']} ({top.get('Release Year', '?')})]")
    print(f"[Prompt snippet: {prompt[:120]}...]\n")

    return generate(prompt, max_new_tokens=max_new_tokens,
                    temperature=temperature, top_k=top_k)
    

print(chat("sweet and low"))

